## Импорты и загрузка переменных окружения

In [1]:
import os
import pandas as pd
import psycopg
import mlflow
from dotenv import load_dotenv
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    confusion_matrix,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    log_loss,
)

load_dotenv()

/home/mle-user/projects/mle-mlflow/.venv/lib/python3.10/site-packages/pydantic/_internal/_config.py:408: UserWarning: Valid config keys have changed in V2:
* 'schema_extra' has been renamed to 'json_schema_extra'
  warnings.warn(message, UserWarning)


True

## Константы и загрузка данных из БД

In [2]:
TABLE_NAME = "users_churn"

TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000

EXPERIMENT_NAME = "Hyperparameter_tuning_lesson_3"
RUN_NAME = "model_grid_search"
REGISTRY_MODEL_NAME = "hyperparameter_tuning_model"

connection = {"sslmode": "require", "target_session_attrs": "read-write"}
postgres_credentials = {
    "host": os.getenv("DB_DESTINATION_HOST"),
    "port": os.getenv("DB_DESTINATION_PORT"),
    "dbname": os.getenv("DB_DESTINATION_NAME"),
    "user": os.getenv("DB_DESTINATION_USER"),
    "password": os.getenv("DB_DESTINATION_PASSWORD"),
    "sslmode": "disable",
}
connection.update(postgres_credentials)

with psycopg.connect(**connection) as conn:
    with conn.cursor() as cur:
        cur.execute(f"SELECT * FROM destination.{TABLE_NAME}")
        data = cur.fetchall()
        columns = [col.name for col in cur.description]

df = pd.DataFrame(data, columns=columns)

# Минимальная очистка пропусков в числовом признаке
df["total_charges"] = df["total_charges"].fillna(0)

## Разбиение выборки и подбор гиперпараметров

In [3]:
features = ["monthly_charges", "total_charges", "senior_citizen"]
target = "target"

split_column = "begin_date"
stratify_column = "target"
test_size = 0.2

df = df.sort_values(by=[split_column])

X_train, X_test, y_train, y_test = train_test_split(df[features], df[target], test_size=test_size, shuffle=False)

print(f"Размер выборки для обучения: {X_train.shape}")
print(f"Размер выборки для теста: {X_test.shape}")

loss_function = "Logloss"
task_type = "CPU"
random_seed = 0
iterations = 300
verbose = False

params = {
    "depth": list((3, 5, 7)),
    "learning_rate": [0.01, 0.05, 0.1],
    "l2_leaf_reg": list((1, 5, 10)),
}

model = CatBoostClassifier(
    loss_function=loss_function,
    task_type=task_type,
    random_seed=random_seed,
    iterations=iterations,
    verbose=verbose,
)

cv = GridSearchCV(
    estimator=model,
    param_grid=params,
    cv=2,
    scoring="roc_auc",
    n_jobs=-1
)

clf = cv.fit(X_train, y_train)

Размер выборки для обучения: (5634, 3)
Размер выборки для теста: (1409, 3)


## Расчёт метрик GridSearchCV и логирование модели в MLflow

In [4]:
os.environ["MLFLOW_S3_ENDPOINT_URL"] = "https://storage.yandexcloud.net"
os.environ["S3_BUCKET_NAME"] = os.getenv("S3_BUCKET_NAME")
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("AWS_SECRET_ACCESS_KEY")

mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")
mlflow.set_registry_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")

cv_results = pd.DataFrame(clf.cv_results_)
best_params = clf.best_params_

model_best = CatBoostClassifier(
    **best_params,
    loss_function=loss_function,
    task_type=task_type,
    random_seed=random_seed,
    iterations=iterations,
    verbose=verbose,
)
model_best.fit(X_train, y_train)

prediction = model_best.predict(X_test)
probas = model_best.predict_proba(X_test)[:, 1]

metrics = {}
_, err1, _, err2 = confusion_matrix(y_test, prediction, normalize="all").ravel()
auc = roc_auc_score(y_test, probas)
precision = precision_score(y_test, prediction)
recall = recall_score(y_test, prediction)
f1 = f1_score(y_test, prediction)
logloss = log_loss(y_test, probas)

metrics["err1"] = err1
metrics["err2"] = err2
metrics["auc"] = auc
metrics["precision"] = precision
metrics["recall"] = recall
metrics["f1"] = f1
metrics["logloss"] = logloss

best_idx = clf.best_index_
metrics["mean_fit_time"] = cv_results.loc[best_idx, "mean_fit_time"]
metrics["std_fit_time"] = cv_results.loc[best_idx, "std_fit_time"]
metrics["mean_test_score"] = cv_results.loc[best_idx, "mean_test_score"]
metrics["std_test_score"] = cv_results.loc[best_idx, "std_test_score"]
metrics["best_score"] = clf.best_score_

pip_requirements = ["catboost", "scikit-learn", "pandas"]
signature = mlflow.models.infer_signature(X_test, prediction)
input_example = X_test[:10]

experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
experiment_id = (
    experiment.experiment_id
    if experiment
    else mlflow.create_experiment(EXPERIMENT_NAME)
)

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    mlflow.log_metrics(metrics)
    mlflow.log_params(best_params)

    cv_info = mlflow.sklearn.log_model(cv, artifact_path='cv')
    
    model_info = mlflow.catboost.log_model(
        cb_model=model_best,
        artifact_path="models",
        registered_model_name=REGISTRY_MODEL_NAME,
        signature=signature,
        input_example=input_example,
        pip_requirements=pip_requirements,
    )

/home/mle-user/projects/mle-mlflow/.venv/lib/python3.10/site-packages/mlflow/models/signature.py:212: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  inputs = _infer_schema(model_input) if model_input is not None else None
Registered model 'hyperparameter_tuning_model' already exists. Creating a new version of this model...
2026/11/20 06:56:13 INFO mlflow.tracking._model_regi

## Случайный поиск гиперпараметров (RandomizedSearchCV) и логирование в MLflow

In [6]:
import numpy as np
from sklearn.model_selection import RandomizedSearchCV

RUN_NAME = "model_random_search"

params = {
    "depth": list(range(3, 8)),
    "learning_rate": np.logspace(-3, -1, 5),
    "l2_leaf_reg": np.logspace(-2, 1, 5),
}

model = CatBoostClassifier(
    loss_function=loss_function,
    task_type=task_type,
    random_seed=random_seed,
    iterations=iterations,
    verbose=verbose,
)

cv = RandomizedSearchCV(
    estimator=model,
    param_distributions=params,
    n_iter=20,
    cv=2,
    scoring="roc_auc",
    random_state=random_seed,
    n_jobs=-1,
)

clf = cv.fit(X_train, y_train)

cv_results = pd.DataFrame(clf.cv_results_)
best_params = clf.best_params_

model = CatBoostClassifier(
    **best_params,
    loss_function=loss_function,
    task_type=task_type,
    random_seed=random_seed,
    iterations=iterations,
    verbose=verbose,
)
model.fit(X_train, y_train)

prediction = model.predict(X_test)
probas = model.predict_proba(X_test)[:, 1]

metrics = {}
_, err1, _, err2 = confusion_matrix(y_test, prediction, normalize="all").ravel()
auc = roc_auc_score(y_test, probas)
precision = precision_score(y_test, prediction)
recall = recall_score(y_test, prediction)
f1 = f1_score(y_test, prediction)
logloss = log_loss(y_test, probas)

metrics["err1"] = err1
metrics["err2"] = err2
metrics["auc"] = auc
metrics["precision"] = precision
metrics["recall"] = recall
metrics["f1"] = f1
metrics["logloss"] = logloss

best_idx = clf.best_index_
metrics["mean_fit_time"] = cv_results.loc[best_idx, "mean_fit_time"]
metrics["std_fit_time"] = cv_results.loc[best_idx, "std_fit_time"]
metrics["mean_test_score"] = cv_results.loc[best_idx, "mean_test_score"]
metrics["std_test_score"] = cv_results.loc[best_idx, "std_test_score"]
metrics["best_score"] = clf.best_score_

signature = mlflow.models.infer_signature(X_test, prediction)
input_example = X_test[:10]

experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    mlflow.log_metrics(metrics)
    mlflow.log_params(best_params)

    cv_info = mlflow.sklearn.log_model(cv, artifact_path="cv")

    model_info = mlflow.catboost.log_model(
        cb_model=model,
        artifact_path="models",
        registered_model_name=REGISTRY_MODEL_NAME,
        signature=signature,
        input_example=input_example,
        pip_requirements=pip_requirements,
    )

/home/mle-user/projects/mle-mlflow/.venv/lib/python3.10/site-packages/mlflow/models/signature.py:212: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  inputs = _infer_schema(model_input) if model_input is not None else None
Registered model 'hyperparameter_tuning_model' already exists. Creating a new version of this model...
2026/11/20 07:04:28 INFO mlflow.tracking._model_regi